In [45]:
import numpy as np
from stapl3d.preprocessing import shading
import matplotlib.pyplot as plt
import argparse
from matplotlib import pyplot as plt
import matplotlib
import random
import os
import cv2
from PIL import Image, ImageEnhance
from skimage import io
from skimage import color
from skimage import exposure
from readlif.reader import LifFile

In [38]:
filepath = "data/flim_sirDNA_cellbrite650_zsatck.lif"
            
lif = LifFile(filepath)

lif_ch1 = lif.get_image(0)
lif_ch2 = lif.get_image(1)
lif_ch3 = lif.get_image(2)
lif_ch4 = lif.get_image(4)
lif_ch5 = lif.get_image(5)
lif_ch6 = lif.get_image(6)
lif_ch7 = lif.get_image(7)
lif_ch8 = lif.get_image(8)
lif_ch9 = lif.get_image(9)

In [39]:
lif_ch9

'LifImage object with dimensions: Dims(x=512, y=300, z=1, t=1, m=1)'

In [40]:
ch1,ch2,ch3 = 0,1,2

In [41]:
l_layer,u_layer = 1, 36

In [42]:
dstack = []

In [43]:
selected_channels = [ch1,ch2,ch3]

#For each channel in the 3Dstack
for i in selected_channels:
    
    #Temporary list holding the collected stacked planes from channel "i" each iteration
    tmp_channel_stacked_planes = []
    
    #For each layer in the range of lower and uper layer limits
    for j in range(l_layer,u_layer):

        # Definimos una lista que guarda los datos
        lif_channel = []
        m_idx = 3
        
        # Leemos el archivo .lif
        lif = LifFile(filepath)
        
        # Tomamos la primera imagen del archivo .lif
        lim = lif.get_image(i)
        #frame = lim.get_frame(z=j, c=i, t=0, m=m_idx)
        
        # Obtenemos la cantidad de tiles en la dimensión m (m_idx)
        n_tiles = lim.dims[m_idx]
        print(i)
        # Iteramos tile por tile
        for m in range(n_tiles):
        
            # Obtenemos el frame para nuestros canales i, plano j (l y u_layer), tiempo 0 y tile m
            data = lim.get_frame(z=j, t=0, m=m)
        
            # Agregamos el frame a la lista dstack
            dstack.append(data)

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2


In [48]:
#Collects all planes from all 3 channels
channel_stacks = []
#List containing the calculated percentile value from each channel from the stack 
percentiles = []

In [49]:
#LOADING THE CHANNEL DATA FROM THE 3D STACK AND CALCULATING PERCENTILE FOR EACH CHANNEL
selected_channels = [ch1,ch2,ch3]
#For each channel in the 3Dstack
for i in selected_channels:
    #Temporary list holding the collected stacked planes from channel "i" each iteration
    tmp_channel_stacked_planes = []
    #For each layer in the range of lower and uper layer limits
    for j in range(l_layer,u_layer):

        # Si el archivo es .lif...
        if filepath.endswith('.lif'):

            # Definimos una lista que guarda los datos
            lif_channel = []
            m_idx = 3
            
            # Leemos el archivo .lif
            lif = LifFile(filepath)
            
            # Tomamos la primera imagen del archivo .lif
            lim = lif.get_image(i)
            #frame = lim.get_frame(z=j, c=i, t=0, m=m_idx)
            
            # Obtenemos la cantidad de tiles en la dimensión m (m_idx)
            n_tiles = lim.dims[m_idx]
            print(i)
            # Iteramos tile por tile
            for m in range(n_tiles):
            
                # Obtenemos el frame para nuestros canales i, plano j (l y u_layer), tiempo 0 y tile m
                data = lim.get_frame(z=j, t=0, m=m)
            
                # Agregamos el frame a la lista dstack
                dstack.append(data)
        
        else:
                dstack = shading.read_tiled_plane(filepath,i,j)
            
        #Stacking the 49 tiles on top of each other. 
        dstacked = np.stack(dstack, axis=0)
        #Add stacked tiles from the single plane to temp list collecting each plane
        tmp_channel_stacked_planes.append(dstacked)

    #Stacking all collected planes from a single channel as the following dimensions l,y,x   
    planes_stacked = np.vstack(tmp_channel_stacked_planes)
    #Append stacked planes of a single channel to list which collects all channel planes. 
    channel_stacks.append(planes_stacked)
    #Making a 1D vector for percentile calculation
    flat_stacked_planes = planes_stacked.flatten()
    #Calculating percentile value for the channel "i"
    p_val = np.percentile(planes_stacked, percentile)

    #Append percentile values to list
    percentiles.append(p_val)

#AUTOMATING PATCH EXTRACTION
#Get dimensions that will be used to extract patches
dimensions = channel_stacks[0].shape 


#Getting dimension of 3D stack l=layers, y and x tile size
l = dimensions[0]
y = dimensions[1]
x = dimensions[2]

#Upperbound for x and y (avoid taking patch beyond frame size)
ubx = x-patchsize
uby = y-patchsize

#Defining proportions of the dataset
t_train = int(datasize)
t_test  = int(t_train*0.2)
t_val   = int(t_train*0.2)


#----------****CREATING FOLDERS****----------
if alpha:
    wb = str(alpha)+"_"+str(round(1-alpha,2))
    datafoldername = biosample+"_"+str(datasize)+"_"+Norm+"_"+mode+"_"+wb+"_patches"

elif Brightness:
    datafoldername = biosample+"_"+str(datasize)+"_"+Norm+"_"+mode+"_br"+Brightness+"%"+"_patches"
else:
    datafoldername = biosample+"_"+str(datasize)+"_"+Norm+"_"+mode+"_patches"


os.mkdir(datafoldername)
if os.path.exists(datafoldername+"/train")==False:
    os.mkdir(datafoldername+"/train")
if os.path.exists(datafoldername+"/test")==False:
    os.mkdir(datafoldername+"/test")
if os.path.exists(datafoldername+"/val")==False:
    os.mkdir(datafoldername+"/val")    


#----------********----------
#Extracting patches for training testing and validation

#Generating training set
for t in range(t_train):    

    #Random patch is extracted and a synthetic source image and a ground truth target image is returned concatenated in AB png format
    image_AB = get_patch_train(l,uby,ubx,patchsize,channels,channel_stacks,percentiles,mode,alpha,Norm,Brightness)
    #Saving the AB patch
    matplotlib.image.imsave(datafoldername+"/train/"+str(t)+'.png', image_AB.astype(np.uint8))

for t in range(t_test):  
    #Random patch is extracted and a synthetic source image and a ground truth target image is returned concatenated in AB png format
    image_AB = get_patch_train(l,uby,ubx,patchsize,channels,channel_stacks,percentiles,mode,alpha,Norm,Brightness)
    #Saving the AB patch
    matplotlib.image.imsave(datafoldername+"/test/"+str(t)+'.png', image_AB.astype(np.uint8))

for t in range(t_val):  
    #Random patch is extracted and a synthetic source image and a ground truth target image is returned concatenated in AB png format
    image_AB = get_patch_train(l,uby,ubx,patchsize,channels,channel_stacks,percentiles,mode,alpha,Norm,Brightness)
    #Saving the AB patch
    matplotlib.image.imsave(datafoldername+"/val/"+str(t)+'.png', image_AB.astype(np.uint8))

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


NameError: name 'percentile' is not defined